This notebook takes care of:

1. Reading the cleaned ads file
2. Creating a "balanced" test set with:
    - descriptions longer than 200 characters (lower than that little to no information is relevant)
    - 3 buckets for description length (so we check short, medium and long ads)
    - 10 jobs per each month (Jan-Aug)
3. Saving the clean test set, ready to be labelled
4. Saving the training set

In [ ]:
import pandas as pd
import json

# clean_ads is not available so for now I test on raw_ads_2511 imagining it to be the clean df
with open("../data/raw_ads_2511.json", encoding="utf-8") as f:
    df_clean = pd.DataFrame(json.load(f))

print("Rows in full clean df:", len(df_clean))

# the clean df is already deduplicated on id but worth to check
df = df_clean.drop_duplicates(subset="id", keep="last").copy()
print("Rows in df after dropping duplicates:", len(df))

# Test set

In [ ]:
# checking ads with short descriptions
shorty = df[df["description"].str.len() < 200]

print("Number of short descriptions:", len(shorty))
shorty

In [ ]:
# remove jobs with descriptions shorter than 200 characters
df = df[df["description"].str.len() >= 200]

print("Number of rows after dropping short description ads:", len(df))

In [ ]:
# restrict to the analysis window: Jan-Aug of both years
df["published"] = pd.to_datetime(df["published"])

df["year"] = df["published"].dt.year
df["month"] = df["published"].dt.month
df = df[(df["month"] <= 8) & (df["year"].isin([2025, 2026]))].copy()

print("Number of rows after selection of relevant months:", len(df))
df.head()

In [ ]:
# length buckets
df["len_bucket"] = pd.qcut(df["description"].str.len(), 3, labels=["short", "medium", "long"])

# shuffles the ads
df = df.sample(frac=1, random_state=42)

df.head()

In [ ]:
# helpfunction to select equal number of ads per month
def sample_per_month(d, n_per_month=10, seed=42):
    return d.groupby("month").sample(
        n=n_per_month, random_state=seed
    )

In [ ]:
# creating the test set and checking it
test_set = pd.concat([sample_per_month(df[df.year == y]) for y in (2025, 2026)])

print("Total ads:", len(test_set))
print("-" * 50)
print("\nNumber of ads per month and year:")
print(test_set.groupby(["year", "month"]).size().unstack(0))
print("-" * 50)
print("\nNumber of ads per lenght bucket per month and year:")
print(test_set.groupby(["year", "len_bucket"], observed=True).size().unstack(0))
print("-" * 50)
test_set.head(15)

In [ ]:
# The test set only need "id" and "description", so I drop the rest of the cols
test_set = test_set[["id", "description"]].reset_index(drop=True)
test_set.head()

In [ ]:
# Saving test set
with open("../data/test_set.json", "w", encoding="utf-8") as f:
    json.dump(test_set.to_dict(orient="records"), f, ensure_ascii=False, indent=2)

# Training set

The training set should not contain ads from the test set, so the are removed and the training set is saved.

In [ ]:
train_set = df_clean[~df_clean["id"].isin(test_set["id"])].reset_index(drop=True)

print("Full:", len(df))
print("Test:", len(test_set))
print("Train:", len(train_set))
train_set.head()

In [ ]:
# Saving test set
with open("../data/train_set.json", "w", encoding="utf-8") as f:
    json.dump(train_set.to_dict(orient="records"), f, ensure_ascii=False, indent=2)